# 03 Dataset Preprocessing

## Purpose

This notebook tokenizes the fixed train, validation, and test splits for one tokenizer setting and exports PyTorch-ready tensors.

## Inputs

- `MyDrive/ProjectRoot2/data/splits/train.txt`
- `MyDrive/ProjectRoot2/data/splits/val.txt`
- `MyDrive/ProjectRoot2/data/splits/test.txt`
- tokenizer files from one folder in `MyDrive/ProjectRoot2/tokenizers/`

## Outputs

- `train_dataset.pt`
- `val_dataset.pt`
- `test_dataset.pt`
- `preprocessing_summary.json`
- `tokenization_preview.csv`

## Notes to myself

This is the handoff notebook between tokenizer generation and model training. I want the output folder to tell me exactly which tokenizer setting produced the tensors that go into pretraining.

## Setup note

Same pattern again.

- code and notebooks stay in GitHub
- tokenized datasets stay in Drive
- Colab pulls the repo at the start
- the final cell syncs the notebook back to GitHub

In [8]:
# ==============================================================================
# 0. SET UP THE COLAB ENVIRONMENT
# ==============================================================================
import os
import sys

from google.colab import drive, userdata

drive.mount('/content/drive')

GITHUB_USER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    print('Repository already exists. Pulling latest changes...')

%cd {REPO_DIR}

!git config --global user.email "hb791-dev@users.noreply.github.com"
!git config --global user.name "hb791-dev"
!git config --global pull.rebase false
!git pull {REPO_URL} main --no-edit -q

if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

print('Colab environment ready.')
print(f'Repo directory: {REPO_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repository already exists. Pulling latest changes...
/content/glycan-roberta
Colab environment ready.
Repo directory: /content/glycan-roberta


## Active tokenizer configuration

This is the one thing I should change on purpose when I switch tokenizer settings. I'm pointing the notebook to one tokenizer folder and one output folder so the exported tensors stay organized.

In [9]:
# ==============================================================================
# 1. DEFINE THE ACTIVE TOKENIZER CONFIGURATION
# ==============================================================================
PROJECT_ROOT = '/content/drive/MyDrive/ProjectRoot2'

TOKENIZER_FAMILY = 'manual'   # 'byte_bpe', 'manual', or 'hybrid_char_bpe'
SETTING_LABEL = 'v1_train_only'               # examples: 'v300_m2', 'v1_train_only', 'v70_m2'

SPLITS_DIR = os.path.join(PROJECT_ROOT, 'data', 'splits')
TOKENIZER_DIR = os.path.join(PROJECT_ROOT, 'tokenizers', TOKENIZER_FAMILY, SETTING_LABEL)
OUTPUT_DATASET_DIR = os.path.join(PROJECT_ROOT, 'tokenized_datasets', TOKENIZER_FAMILY, SETTING_LABEL)

TRAIN_PATH = os.path.join(SPLITS_DIR, 'train.txt')
VAL_PATH = os.path.join(SPLITS_DIR, 'val.txt')
TEST_PATH = os.path.join(SPLITS_DIR, 'test.txt')

os.makedirs(OUTPUT_DATASET_DIR, exist_ok=True)

print('Tokenizer family:')
print(TOKENIZER_FAMILY)
print('\nSetting label:')
print(SETTING_LABEL)
print('\nTokenizer directory:')
print(TOKENIZER_DIR)
print('\nOutput dataset directory:')
print(OUTPUT_DATASET_DIR)

for required_path in [TRAIN_PATH, VAL_PATH, TEST_PATH, TOKENIZER_DIR]:
    if not os.path.exists(required_path):
        raise FileNotFoundError(f'Required path not found: {required_path}')

Tokenizer family:
manual

Setting label:
v1_train_only

Tokenizer directory:
/content/drive/MyDrive/ProjectRoot2/tokenizers/manual/v1_train_only

Output dataset directory:
/content/drive/MyDrive/ProjectRoot2/tokenized_datasets/manual/v1_train_only


## Load the tokenizer and inspect a few examples

I want one quick check here before exporting tensors. If the tokenizer is clearly splitting things in a weird way, I'd rather catch that now than after I start training.

In [10]:
# ==============================================================================
# 2. LOAD THE TOKENIZER AND PREVIEW TOKENIZATION
# ==============================================================================
import random
from collections import Counter

import pandas as pd
from transformers import PreTrainedTokenizerFast

with open(TRAIN_PATH, 'r', encoding='utf-8') as file:
    train_sequences = [line.strip() for line in file if line.strip()]

tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_DIR)
print(f'Loaded tokenizer from: {TOKENIZER_DIR}')
print(f'Vocabulary size: {len(tokenizer)}')

random.seed(42)
sample_sequences = random.sample(train_sequences, 3) if len(train_sequences) >= 3 else train_sequences

all_tokens = []
preview_rows = []

for sample_index, sequence in enumerate(sample_sequences, start=1):
    token_ids = tokenizer.encode(sequence, add_special_tokens=False)
    tokens = tokenizer.convert_ids_to_tokens(token_ids)

    all_tokens.extend(tokens)
    preview_rows.append(
        {
            'sample_index': sample_index,
            'sequence': sequence,
            'num_tokens': len(tokens),
            'tokens': ' | '.join(tokens[:40]),
        }
    )

preview_df = pd.DataFrame(preview_rows)
display(preview_df)

print('\nTop tokens in the preview set')
for token, count in Counter(all_tokens).most_common(10):
    print(f'{token:<20} : {count}')

Loaded tokenizer from: /content/drive/MyDrive/ProjectRoot2/tokenizers/manual/v1_train_only
Vocabulary size: 78


,sample_index,sequence,num_tokens,tokens
0,1,Galb1-4GlcNAcb1-2Mana1-3(Galb1-4GlcNAcb1-2(Fuc...,32,Gal | b1-4 | GlcNAc | b1-2 | Man | a1-3 | ( | ...
1,2,NeuAca2-3Galb1-3(NeuAca2-6)Gal,9,NeuAc | a2-3 | Gal | b1-3 | ( | NeuAc | a2-6 |...
2,3,NeuAc?2-?Gal?1-?(Fuc?1-?Gal?1-?GlcNAc?1-?)GalNAc,13,NeuAc | ?2-? | Gal | ?1-? | ( | Fuc | ?1-? | G...



Top tokens in the preview set
Gal                  : 7
GlcNAc               : 6
b1-4                 : 5
(                    : 5
)                    : 5
NeuAc                : 4
?1-?                 : 4
Man                  : 3
b1-2                 : 2
a1-3                 : 2


## Choose a padded sequence length

I'm using the training split to estimate a practical max length. The point is to avoid padding every sequence to the single longest outlier while still keeping almost all of the data intact.

In [11]:
# ==============================================================================
# 3. COMPUTE TOKEN LENGTH STATISTICS AND SELECT MAX LENGTH
# ==============================================================================
import numpy as np

train_token_lengths = np.array(
    [len(tokenizer.encode(sequence, add_special_tokens=False)) for sequence in train_sequences]
)

# Add BOS and EOS because they are included in the exported tensors.
train_total_lengths = train_token_lengths + 2

max_observed_length = int(train_total_lengths.max())
p95_length = float(np.percentile(train_total_lengths, 95))
p99_length = float(np.percentile(train_total_lengths, 99))
selected_max_length = int(((int(p99_length) + 7) // 8) * 8)

length_summary = {
    'max_observed_length': max_observed_length,
    'p95_length': p95_length,
    'p99_length': p99_length,
    'selected_max_length': selected_max_length,
}

length_summary_df = pd.DataFrame(
    {
        'metric': list(length_summary.keys()),
        'value': list(length_summary.values()),
    }
)

display(length_summary_df)


,metric,value
0,max_observed_length,104.0
1,p95_length,42.0
2,p99_length,51.0
3,selected_max_length,56.0


## Export the tokenized datasets

This is the main preprocessing step. Each split gets converted to padded `input_ids` and `attention_mask` tensors using the selected tokenizer and the chosen max length.

In [12]:
# ==============================================================================
# 4. TOKENIZE EACH SPLIT AND BUILD PYTORCH DATA STRUCTURES
# ==============================================================================
import torch

def load_sequences(path):
    with open(path, 'r', encoding='utf-8') as file:
        return [line.strip() for line in file if line.strip()]

def process_dataset(filepath, max_seq_len):
    sequences = load_sequences(filepath)

    input_ids_matrix = []
    attention_mask_matrix = []
    truncated_count = 0

    pad_id = tokenizer.pad_token_id
    bos_id = tokenizer.bos_token_id
    eos_id = tokenizer.eos_token_id

    for sequence in sequences:
        token_ids = tokenizer.encode(sequence, add_special_tokens=False)
        sequence_ids = [bos_id] + token_ids + [eos_id]

        if len(sequence_ids) > max_seq_len:
            sequence_ids = sequence_ids[:max_seq_len]
            attention_mask = [1] * max_seq_len
            truncated_count += 1
        else:
            pad_length = max_seq_len - len(sequence_ids)
            attention_mask = [1] * len(sequence_ids) + [0] * pad_length
            sequence_ids = sequence_ids + [pad_id] * pad_length

        input_ids_matrix.append(sequence_ids)
        attention_mask_matrix.append(attention_mask)

    dataset = {
        'input_ids': torch.tensor(input_ids_matrix, dtype=torch.long),
        'attention_mask': torch.tensor(attention_mask_matrix, dtype=torch.long),
    }

    summary = {
        'num_sequences': len(sequences),
        'num_truncated': truncated_count,
        'tensor_shape': list(dataset['input_ids'].shape),
    }

    return dataset, summary

train_dataset, train_summary = process_dataset(TRAIN_PATH, selected_max_length)
val_dataset, val_summary = process_dataset(VAL_PATH, selected_max_length)
test_dataset, test_summary = process_dataset(TEST_PATH, selected_max_length)

split_tensor_summary = pd.DataFrame(
    [
        {'split': 'train', **train_summary},
        {'split': 'val', **val_summary},
        {'split': 'test', **test_summary},
    ]
)

display(split_tensor_summary)


,split,num_sequences,num_truncated,tensor_shape
0,train,17453,52,"[17453, 56]"
1,val,2182,8,"[2182, 56]"
2,test,2182,6,"[2182, 56]"


## Save the exported files

I want the tensor files plus a small metadata record in the same folder. That way I can tell later what tokenizer and max length produced the datasets without opening the notebook.

In [13]:
# ==============================================================================
# 5. SAVE THE TOKENIZED DATASETS AND SUMMARY FILES
# ==============================================================================
import json

train_export_path = os.path.join(OUTPUT_DATASET_DIR, 'train_dataset.pt')
val_export_path = os.path.join(OUTPUT_DATASET_DIR, 'val_dataset.pt')
test_export_path = os.path.join(OUTPUT_DATASET_DIR, 'test_dataset.pt')

torch.save(train_dataset, train_export_path)
torch.save(val_dataset, val_export_path)
torch.save(test_dataset, test_export_path)

preview_export_path = os.path.join(OUTPUT_DATASET_DIR, 'tokenization_preview.csv')
preview_df.to_csv(preview_export_path, index=False)

summary_payload = {
    'tokenizer_family': TOKENIZER_FAMILY,
    'setting_label': SETTING_LABEL,
    'tokenizer_dir': TOKENIZER_DIR,
    'selected_max_length': selected_max_length,
    'length_summary': length_summary,
    'train_summary': train_summary,
    'val_summary': val_summary,
    'test_summary': test_summary,
    'saved_files': [
        'train_dataset.pt',
        'val_dataset.pt',
        'test_dataset.pt',
        'tokenization_preview.csv',
        'preprocessing_summary.json',
    ],
}

summary_export_path = os.path.join(OUTPUT_DATASET_DIR, 'preprocessing_summary.json')
with open(summary_export_path, 'w', encoding='utf-8') as file:
    json.dump(summary_payload, file, indent=2)

print('Tokenized datasets saved.')
print(f'Output folder: {OUTPUT_DATASET_DIR}')
print(f'Summary file: {summary_export_path}')

Tokenized datasets saved.
Output folder: /content/drive/MyDrive/ProjectRoot2/tokenized_datasets/manual/v1_train_only
Summary file: /content/drive/MyDrive/ProjectRoot2/tokenized_datasets/manual/v1_train_only/preprocessing_summary.json


## GitHub sync note

Same idea as the earlier notebooks. The tensor files stay in Drive. The notebook itself stays versioned in GitHub.

In [14]:
# ==============================================================================
# 6. SAVE THE NOTEBOOK BACK TO GITHUB
# ==============================================================================
REPO_NOTEBOOK_PATH = os.path.join(REPO_DIR, 'notebooks', '03_dataset_preprocessing.ipynb')
DRIVE_NOTEBOOK_PATH = '/content/drive/MyDrive/Colab Notebooks/03_dataset_preprocessing.ipynb'

if os.path.exists(DRIVE_NOTEBOOK_PATH):
    !cp "{DRIVE_NOTEBOOK_PATH}" "{REPO_NOTEBOOK_PATH}"

    try:
        with open(REPO_NOTEBOOK_PATH, 'r', encoding='utf-8') as file:
            notebook_json = json.load(file)

        if 'widgets' in notebook_json.get('metadata', {}):
            del notebook_json['metadata']['widgets']

        with open(REPO_NOTEBOOK_PATH, 'w', encoding='utf-8') as file:
            json.dump(notebook_json, file, indent=1)
    except Exception as exc:
        print(f'Notebook metadata cleanup skipped: {exc}')

    %cd {REPO_DIR}
    !git add notebooks/03_dataset_preprocessing.ipynb
    !git commit -m "Update 03_dataset_preprocessing" || echo "No new changes to commit."
    !git pull {REPO_URL} main --no-edit -q
    !git push {REPO_URL} main -q

    print('Notebook synced to GitHub.')
else:
    print(f'Notebook file not found at: {DRIVE_NOTEBOOK_PATH}')
    print('Save the notebook in Colab, then run this cell again.')

/content/glycan-roberta
[main e23c16e] Update 03_dataset_preprocessing
 1 file changed, 897 insertions(+), 44 deletions(-)
Notebook synced to GitHub.
